## **Notebook 1: `JOB_MARKET_ANALYSIS`**


## **1. Importing libraries**

In [1]:
import pandas as pd
import numpy as np

import re
import os

import matplotlib.pyplot as plt
import seaborn as sns

## **Loading the dataset**

In [2]:
df = pd.read_csv(r"D:\Project 8\all_upwork_jobs_2024-02-07-2024-03-24.csv")
df.head()

,title,link,published_date,is_hourly,hourly_low,hourly_high,budget,country
0,Experienced Media Buyer For Solar Pannel and R...,https://www.upwork.com/jobs/Experienced-Media-...,2024-02-17 09:09:54+00:00,False,NaN,NaN,500.0,NaN
1,Full Stack Developer,https://www.upwork.com/jobs/Full-Stack-Develop...,2024-02-17 09:09:17+00:00,False,NaN,NaN,1100.0,United States
2,SMMA Bubble App,https://www.upwork.com/jobs/SMMA-Bubble-App_%7...,2024-02-17 09:08:46+00:00,True,10.0,30.0,NaN,United States
3,Talent Hunter Specialized in Marketing,https://www.upwork.com/jobs/Talent-Hunter-Spec...,2024-02-17 09:08:08+00:00,True,NaN,NaN,NaN,United States
4,Data Engineer,https://www.upwork.com/jobs/Data-Engineer_%7E0...,2024-02-17 09:07:42+00:00,False,NaN,NaN,650.0,India


## **4. Basic inspection**

In [3]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nInfo:\n")
df.info()
print("\nMissing values:\n", df.isnull().sum())

Shape: (244828, 8)

Columns:
 ['title', 'link', 'published_date', 'is_hourly', 'hourly_low', 'hourly_high', 'budget', 'country']

Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244828 entries, 0 to 244827
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   title           244827 non-null  object 
 1   link            244827 non-null  object 
 2   published_date  244828 non-null  object 
 3   is_hourly       244828 non-null  bool   
 4   hourly_low      102422 non-null  float64
 5   hourly_high     98775 non-null   float64
 6   budget          103891 non-null  float64
 7   country         239751 non-null  object 
dtypes: bool(1), float64(3), object(4)
memory usage: 13.3+ MB

Missing values:
 title                  1
link                   1
published_date         0
is_hourly              0
hourly_low        142406
hourly_high       146053
budget            140937
country             5077
dtype: int64


## **5. Standardizing the column names**

In [4]:
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_")
)
df.columns

Index(['title', 'link', 'published_date', 'is_hourly', 'hourly_low',
       'hourly_high', 'budget', 'country'],
      dtype='object')

## **6. Cleaning title text**

In [5]:
def clean_title(text):
    if pd.isna(text):
        return np.nan
    text = str(text).lower().strip()
    text = re.sub(r'[^a-zA-Z0-9\s/+-]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df["clean_title"] = df["title"].apply(clean_title)

## **7. Converting date column**

In [6]:
df["published_date"] = pd.to_datetime(df["published_date"], errors="coerce")

## **8. Numeric conversions**

In [7]:
numeric_cols = ["hourly_low", "hourly_high", "budget"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## **9. Creating derived columns**

In [8]:
df["job_type"] = np.where(df["is_hourly"] == True, "Hourly", "Fixed")

df["avg_hourly_rate"] = np.where(
    df["is_hourly"] == True,
    (df["hourly_low"] + df["hourly_high"]) / 2,
    np.nan
)

df["year"] = df["published_date"].dt.year
df["month"] = df["published_date"].dt.month
df["month_name"] = df["published_date"].dt.month_name()
df["year_month"] = df["published_date"].dt.to_period("M").astype(str)
df["week"] = df["published_date"].dt.to_period("W").astype(str)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_21668\1663225077.py:12: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df["published_date"].dt.to_period("M").astype(str)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21668\1663225077.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["week"] = df["published_date"].dt.to_period("W").astype(str)


## **10. Cleaning country values**

In [9]:
df["country"] = df["country"].astype(str).str.strip()
df["country"] = df["country"].replace({"nan": np.nan, "": np.nan})

## **11. Handling duplicates**

In [10]:
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())

Duplicates before: 0
Duplicates after: 0


## **12. Missing value checking after cleaning**

In [11]:
df.isnull().sum().sort_values(ascending=False)

hourly_high        146053
avg_hourly_rate    146053
hourly_low         142406
budget             140937
country              5077
title                   1
link                    1
clean_title             1
published_date          0
is_hourly               0
job_type                0
year                    0
month                   0
month_name              0
year_month              0
week                    0
dtype: int64

## **13. Spliting into hourly and fixed jobs**

In [12]:
hourly_jobs = df[df["job_type"] == "Hourly"].copy()
fixed_jobs = df[df["job_type"] == "Fixed"].copy()

## **14. Saving cleaned outputs**

In [13]:
df.to_csv("cleaned_jobs.csv", index=False)
hourly_jobs.to_csv("hourly_jobs.csv", index=False)
fixed_jobs.to_csv("fixed_jobs.csv", index=False)

print("Files saved successfully.")

Files saved successfully.


## **15. Final summary**

In [14]:
print("Total jobs:", len(df))
print("Hourly jobs:", len(hourly_jobs))
print("Fixed jobs:", len(fixed_jobs))
print("Date range:", df["published_date"].min(), "to", df["published_date"].max())

Total jobs: 244828
Hourly jobs: 140937
Fixed jobs: 103891
Date range: 2023-11-02 09:22:02+00:00 to 2024-03-24 14:16:47+00:00


## **`Project_Notebook_Structure`**

```job-market-analysis/
│
├── data/
│   ├── raw/
│   │   └── all_upwork_jobs_2024-02-07-2024-03-24.csv
│   └── processed/
│       ├── cleaned_jobs.csv
│       ├── hourly_jobs.csv
│       ├── fixed_jobs.csv
│       ├── categorized_jobs.csv
│       ├── monthly_category_trends.csv
│       └── recommendation_input.csv
│
├── notebooks/
│   ├── 01_Data_Cleaning_Preprocessing.ipynb
│   ├── 02_Keyword_Salary_Analysis.ipynb
│   ├── 03_Emerging_Job_Categories_Trend_Analysis.ipynb
│   ├── 04_High_Demand_Role_Forecasting.ipynb
│   ├── 05_Country_Hourly_Rate_Comparison.ipynb
│   └── 06_Job_Recommendation_Engine.ipynb
│
├── src/
├── app/
├── docker/
├── reports/
├── requirements.txt
└── README.md```